In [ ]:
import os
import json
import pandas as pd
import numpy as np
from transformers import BertTokenizer
from unicodedata import normalize
from transformers import BertModel, BertTokenizer
import torch
from tokenizers import BertWordPieceTokenizer

In [ ]:
######################## data 불러오기 ##########################################################################

In [ ]:
file_path = r"./024.아동·청소년 상담 데이터\3.개방데이터\1.데이터\Training\02.라벨링데이터\TL_out"

In [ ]:
files = os.listdir(file_path)

In [ ]:
json_data_list = []
for file in files:
    if file.endswith(".json"):  # JSON 파일만 선택
        with open(os.path.join(file_path, file), 'r', encoding='utf-8') as f:
            json_data = json.load(f)  # JSON 파일 읽기
            json_data_list.append(json_data)

In [ ]:
len(json_data_list)

In [ ]:
######################## 1. 학대여부 대화문 추출 ###################################################################

In [ ]:
########### 1.1 질문 + 응답

In [ ]:
normal_group = []       # 정상군
observation_needed = [] # 관찰필요
counseling_needed = []  # 상담필요
abuse_suspected = []    # 학대의심
crisis_child = []       # 응급

In [ ]:
for entry in json_data_list:
    crisis_level = entry['info']['위기단계']
    dialogue = ""
    
    # Concatenate dialogues within each entry for 문항 == 학대여부
    for item in entry['list']:
        if item['문항'] == '학대여부':
            for sub_item in item['list']:
                for audio in sub_item.get("audio", []):
                    if audio["type"] == "Q":
                        dialogue += "Q: " + audio["text"] + " "
                    elif audio["type"] == "A":
                        dialogue += "A: " + audio["text"] + " "
    
    # Append the combined dialogue to the corresponding crisis level list
    if crisis_level == "정상군":
        normal_group.append(dialogue.strip())
    elif crisis_level == "관찰필요":
        observation_needed.append(dialogue.strip())
    elif crisis_level == "상담필요":
        counseling_needed.append(dialogue.strip())
    elif crisis_level == "학대의심":
        abuse_suspected.append(dialogue.strip())
    elif crisis_level == "응급":
        crisis_child.append(dialogue.strip())

In [ ]:
print(f"정상군: {len(normal_group)}")
print(f"관찰필요: {len(observation_needed)}") 
print(f"상담필요: {len(counseling_needed)}") 
print(f"학대의심: {len(abuse_suspected)}") 
print(f"응급: {len(crisis_child)}") 

In [ ]:
normal_group[1:3]

In [ ]:
data_QA = normal_group + observation_needed + counseling_needed + abuse_suspected + crisis_child # 다합친 벡터

In [ ]:
len(data_QA)

In [ ]:
########### 1.2 응답만

In [ ]:
normal_group_A = []       # 정상군
observation_needed_A = [] # 관찰필요
counseling_needed_A = []  # 상담필요
abuse_suspected_A = []    # 학대의심
crisis_child_A = []       # 응급

In [ ]:
for entry in json_data_list:
    crisis_level = entry['info']['위기단계']
    answers_only = ""
    
    # Concatenate only answers within each entry for 문항 == 학대여부
    for item in entry['list']:
        if item['문항'] == '학대여부':
            for sub_item in item['list']:
                for audio in sub_item.get("audio", []):
                    if audio["type"] == "A":  # Only store answers
                        answers_only += audio["text"] + " "
    
    # Append the combined answers to the corresponding crisis level list
    if crisis_level == "정상군":
        normal_group_A.append(answers_only.strip())
    elif crisis_level == "관찰필요":
        observation_needed_A.append(answers_only.strip())
    elif crisis_level == "상담필요":
        counseling_needed_A.append(answers_only.strip())
    elif crisis_level == "학대의심":
        abuse_suspected_A.append(answers_only.strip())
    elif crisis_level == "응급":
        crisis_child_A.append(answers_only.strip())

In [ ]:
data_A = normal_group_A + observation_needed_A + counseling_needed_A + abuse_suspected_A + crisis_child_A # 다합친 벡터

In [ ]:
data_A[1:3]

In [ ]:
######################## 2. 나머지 정보 추출 ###################################################################

In [ ]:
def extract_combined_info(data):
    row = {}
    
    # 1. 'info'에서 객체 정보 추출
    info = data.get('info', {})
    row['성별'] = info.get('성별', '')
    row['나이'] = info.get('나이', '')
    row['학년'] = info.get('학년', '')
    row['유형구분'] = info.get('유형구분', '')
    row['위기단계'] = info.get('위기단계', '')
    row['합계점수'] = info.get('합계점수', 0)
    row['학대의심'] = info.get('학대의심', '')
    row['행동특성 점수'] = info.get('행동특성 점수', 0)
    
    # 2. 'list'에서 학대여부를 제외한 각 항목별 대화 추출
    for entry in data.get('list', []):
        if entry.get('문항') != '학대여부':  # '학대여부' 문항을 제외
            for 항목 in entry.get('list', []):
                항목_name = 항목.get('항목', '')  # 각 항목 이름 (예: 통증, 신체손상 등)

                # Q & A 텍스트 추출
                dialogues = []
                if 'audio' in 항목:  # 'audio' 항목이 존재하는지 확인
                    for audio in 항목.get('audio', []):
                        type_prefix = "Q" if audio.get('type', '') == 'Q' else "A"
                        text = f"{type_prefix}: {audio.get('text', '')}"  # 텍스트가 없을 경우 빈 문자열 처리
                        dialogues.append(text)

                # 대화가 있는 경우 항목 이름을 열로 하여 추가
                row[항목_name] = " ".join(dialogues) if dialogues else ''  # 대화가 없을 경우 빈 문자열

    return row

In [ ]:
all_rows = []
for idx, json_data in enumerate(json_data_list):
    dialogues = extract_combined_info(json_data)
    all_rows.append(dialogues)  

In [ ]:
df_other = pd.DataFrame(all_rows)

In [ ]:
print("열 이름:", df_other.columns.tolist())